In [5]:
import numpy as np
import pandas as pd
import random
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import sklearn.utils
import pickle

# --------------------------
# Global config
# --------------------------
SEED = 0
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

EPOCHS = 100
BATCH_SIZE = 1
KFOLD_SPLITS = 4
REPEATS = 300
CSV_PATH = '../CSN_Viability_Database_2025.csv'
OUTPUT_FILENAME = 'T38_combined_pytorch.pkl'
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------
# Data Loading and Preprocessing
# --------------------------
def load_CSN_data():
    return pd.read_csv(CSV_PATH)

CSN = load_CSN_data()

CSN = CSN.drop([
    'Example ID', 'Source', 'Figure ID', 'Data Provider', 'PI',
    'Date Received', 'Data Measurment Published',
    'Prior Exposure', 'Comments', 'Error'
], axis=1)

CSN_prepared = pd.get_dummies(CSN, dtype=int)
CSN_prepared['Surface Area per Liter'] = (
    CSN_prepared['Surface Area (NMC) (m2/g)'] * CSN_prepared['Concentration (mg/L)']
)
CSN_prepared = CSN_prepared.drop(['Surface Area (NMC) (m2/g)'], axis=1)
CSN_prepared['log Concentration'] = np.log10(CSN_prepared['Concentration (mg/L)'] + 1e-9)
CSN_prepared = CSN_prepared.drop(['Concentration (mg/L)'], axis=1)

CSN_test = CSN_prepared.tail(38)
CSN_train = CSN_prepared.iloc[:-38]

X_test = CSN_test.drop(['Viability_Fraction'], axis=1).values.astype(np.float32)
y_test = CSN_test['Viability_Fraction'].values.astype(np.float32)

# --------------------------
# Define PyTorch model
# --------------------------
class MLP(nn.Module):
    def __init__(self, input_dim):
        super(MLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 5),
            nn.ReLU(),
            nn.Linear(5, 5),
            nn.ReLU(),
            nn.Linear(5, 5),
            nn.ReLU(),
            nn.Linear(5, 1),
            nn.ReLU()  # ensure non-negative output
        )

    def forward(self, x):
        return self.net(x)

# --------------------------
# K-Fold Bagging Function
# --------------------------
def run_kfold_bagging(X, y, X_test, seeds, k=KFOLD_SPLITS):
    kf = KFold(n_splits=k, shuffle=True)
    fold_preds = np.zeros((k, X_test.shape[0]))
    weights = np.zeros(k)

    for i, (train_idx, val_idx) in enumerate(kf.split(X)):
        # Split data
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]

        # Convert to torch tensors
        X_tr_t = torch.tensor(X_tr).to(DEVICE)
        y_tr_t = torch.tensor(y_tr).view(-1, 1).to(DEVICE)
        X_val_t = torch.tensor(X_val).to(DEVICE)
        y_val_t = torch.tensor(y_val).view(-1, 1).to(DEVICE)
        X_test_t = torch.tensor(X_test).to(DEVICE)

        # Set seed and model
        torch.manual_seed(seeds[i])
        model = MLP(X.shape[1]).to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        loss_fn = nn.MSELoss()

        # Training loop
        for epoch in range(EPOCHS):
            model.train()
            for j in range(0, len(X_tr), BATCH_SIZE):
                x_batch = X_tr_t[j:j+BATCH_SIZE]
                y_batch = y_tr_t[j:j+BATCH_SIZE]
                optimizer.zero_grad()
                preds = model(x_batch)
                loss = loss_fn(preds, y_batch)
                loss.backward()
                optimizer.step()

        # Validation MAE
        model.eval()
        with torch.no_grad():
            val_preds = model(X_val_t).cpu().numpy().flatten()
            val_mae = np.mean(np.abs(val_preds - y_val))

            test_preds = model(X_test_t).cpu().numpy().flatten()
            fold_preds[i] = test_preds
            weights[i] = 1.0 / (val_mae + 1e-8)

    weighted_pred = np.average(fold_preds, axis=0, weights=weights)
    weighted_std_err = np.sqrt(np.average((fold_preds - weighted_pred)**2, axis=0, weights=weights)) / np.sqrt(k)

    return weighted_pred, weighted_std_err

# --------------------------
# Bagging Repeats
# --------------------------
results = np.zeros((REPEATS, 2, X_test.shape[0]))

X_full = CSN_train.drop(['Viability_Fraction'], axis=1).values.astype(np.float32)
y_full = CSN_train['Viability_Fraction'].values.astype(np.float32)

for run_idx in range(REPEATS):
    shuffled = sklearn.utils.shuffle(CSN_train, random_state=run_idx)
    X_train = shuffled.drop(['Viability_Fraction'], axis=1).values.astype(np.float32)
    y_train = shuffled['Viability_Fraction'].values.astype(np.float32)

    # Standardization
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    seeds = np.random.randint(0, 100000, size=KFOLD_SPLITS)

    mean_pred, std_err = run_kfold_bagging(X_train_scaled, y_train, X_test_scaled, seeds)

    results[run_idx, 0] = mean_pred
    results[run_idx, 1] = std_err

    print(f"[{run_idx+1}/{REPEATS}] completed.")

# --------------------------
# Save Output
# --------------------------
with open(OUTPUT_FILENAME, 'wb') as f:
    pickle.dump(results, f)

print(f"\n✅ Results saved to '{OUTPUT_FILENAME}'")


[1/300] completed.
[2/300] completed.
[3/300] completed.
[4/300] completed.
[5/300] completed.
[6/300] completed.
[7/300] completed.
[8/300] completed.
[9/300] completed.
[10/300] completed.
[11/300] completed.
[12/300] completed.
[13/300] completed.
[14/300] completed.
[15/300] completed.
[16/300] completed.
[17/300] completed.
[18/300] completed.
[19/300] completed.
[20/300] completed.
[21/300] completed.
[22/300] completed.
[23/300] completed.
[24/300] completed.
[25/300] completed.
[26/300] completed.
[27/300] completed.
[28/300] completed.
[29/300] completed.
[30/300] completed.
[31/300] completed.
[32/300] completed.
[33/300] completed.
[34/300] completed.
[35/300] completed.
[36/300] completed.
[37/300] completed.
[38/300] completed.
[39/300] completed.
[40/300] completed.
[41/300] completed.
[42/300] completed.
[43/300] completed.
[44/300] completed.
[45/300] completed.
[46/300] completed.
[47/300] completed.
[48/300] completed.
[49/300] completed.
[50/300] completed.
[51/300] 

In [6]:
# --------------------------
# MAE Statistics over REPEATS
# --------------------------
# Compute MAE per run
mae_list = []

for i in range(REPEATS):
    preds = results[i, 0]  # predicted mean
    mae = np.mean(np.abs(preds - y_test))
    mae_list.append(mae)

mae_array = np.array(mae_list)
mean_mae = np.mean(mae_array)
std_mae = np.std(mae_array)
min_mae = np.min(mae_array)
max_mae = np.max(mae_array)

# Print summary
print("\nSummary of Neural Network predictions over {} random seeds:".format(REPEATS))
print("Mean MAE: {:.4f} ± {:.4f}".format(mean_mae, std_mae))
print("Min MAE: {:.4f}".format(min_mae))
print("Max MAE: {:.4f}".format(max_mae))



Summary of Neural Network predictions over 300 random seeds:
Mean MAE: 0.4431 ± 0.1418
Min MAE: 0.2301
Max MAE: 0.9345
